In [1]:
#============================================
# Celda 1 — Imports y carga del JSON
#============================================
import json
import glob
import pandas as pd
import numpy as np
import os

# Cargar último JSON ambiental
f = sorted(glob.glob('../../data/ambiental/2026/03/*.json'))[-1]
data = json.load(open(f))
print(f'Archivo cargado: {f}')
print(f'Keys disponibles: {list(data.keys())}')


Archivo cargado: ../../data/ambiental/2026/03/30_03_2026.json
Keys disponibles: ['timestamp', 'observacion_meteorologica', 'calidad_aire']


In [2]:
#============================================
# Celda 2 — Explorar estructura AEMET
#============================================
aemet_raw  = data['observacion_meteorologica']
estaciones = aemet_raw['estaciones']

print(f'Total estaciones: {aemet_raw["total_estaciones"]}')
print(f'Timestamp captura: {aemet_raw["timestamp_captura"]}')
print(f'\nKeys de una estación:')
print(list(estaciones[0].keys()))
print(f'\nEjemplo estación[0]:')
print(json.dumps(estaciones[0], indent=2, ensure_ascii=False))


Total estaciones: 10059
Timestamp captura: 2026-03-30T06:57:47.464004

Keys de una estación:
['idema', 'ubi', 'lat', 'lon', 'alt', 'fint', 'ta', 'tamin', 'tamax', 'prec', 'hr', 'vv', 'vmax', 'dv', 'dmax', 'pres', 'pres_nmar', 'vis', 'inso', 'ts', 'tpr']

Ejemplo estación[0]:
{
  "idema": "0009X",
  "ubi": "ALFORJA",
  "lat": 41.213892,
  "lon": 0.963335,
  "alt": 406.0,
  "fint": "2026-03-29T18:00:00+0000",
  "ta": 11.0,
  "tamin": 11.0,
  "tamax": 12.7,
  "prec": 0.0,
  "hr": 22.0,
  "vv": 4.6,
  "vmax": 10.0,
  "dv": 307.0,
  "dmax": 307.0,
  "pres": null,
  "pres_nmar": null,
  "vis": null,
  "inso": null,
  "ts": null,
  "tpr": null
}


In [3]:
#============================================
# Celda 3 — Parsear AEMET a DataFrame
#============================================
df_aemet = pd.DataFrame(estaciones)

# Convertir timestamp
df_aemet['fint'] = pd.to_datetime(df_aemet['fint'], utc=True, errors='coerce')
df_aemet['fecha'] = df_aemet['fint'].dt.date

# Columnas numéricas
cols_num = ['lat','lon','alt','ta','tamin','tamax','prec','hr',
            'vv','vmax','dv','dmax','pres','pres_nmar','vis','inso','ts','tpr']
for c in cols_num:
    if c in df_aemet.columns:
        df_aemet[c] = pd.to_numeric(df_aemet[c], errors='coerce')

print(f'Shape: {df_aemet.shape}')
print(f'Columnas: {list(df_aemet.columns)}')
df_aemet.head()


Shape: (10059, 22)
Columnas: ['idema', 'ubi', 'lat', 'lon', 'alt', 'fint', 'ta', 'tamin', 'tamax', 'prec', 'hr', 'vv', 'vmax', 'dv', 'dmax', 'pres', 'pres_nmar', 'vis', 'inso', 'ts', 'tpr', 'fecha']


,idema,ubi,lat,lon,alt,fint,ta,tamin,tamax,prec,...,vmax,dv,dmax,pres,pres_nmar,vis,inso,ts,tpr,fecha
0,0009X,ALFORJA,41.213892,0.963335,406.0,2026-03-29 18:00:00+00:00,11.0,11.0,12.7,0.0,...,10.0,307.0,307.0,NaN,NaN,NaN,NaN,NaN,NaN,2026-03-29
1,0016A,REUS AEROPUERTO,41.145000,1.163611,71.0,2026-03-29 18:00:00+00:00,15.1,14.9,16.7,0.0,...,10.3,340.0,310.0,1013.1,1022.2,30.0,60.0,14.1,-7.8,2026-03-29
2,0034X,VALLS,41.293053,1.260838,233.0,2026-03-29 18:00:00+00:00,12.7,12.7,14.3,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-03-29
3,0042Y,TARRAGONA FAC. GEOGRAFIA,41.123892,1.249167,55.0,2026-03-29 18:00:00+00:00,14.6,14.6,15.1,0.0,...,8.5,339.0,335.0,NaN,NaN,NaN,NaN,NaN,NaN,2026-03-29
4,0061X,PONTONS,41.417052,1.519269,632.0,2026-03-29 18:00:00+00:00,9.9,9.9,11.9,0.0,...,6.7,318.0,322.0,NaN,NaN,NaN,NaN,NaN,NaN,2026-03-29


In [4]:
#============================================
# Celda 4 — Análisis rápido AEMET
#============================================
print('Valores nulos por columna:')
nulos = df_aemet.isnull().sum()
print(nulos[nulos > 0])

print(f'\nRango temperatura:  {df_aemet["ta"].min():.1f}°C — {df_aemet["ta"].max():.1f}°C')
print(f'Media temperatura:  {df_aemet["ta"].mean():.1f}°C')
print(f'\nEstaciones con precipitación > 0: {(df_aemet["prec"] > 0).sum()}')
print(f'Precipitación máxima: {df_aemet["prec"].max():.1f} mm')
print(f'\nRango altitud: {df_aemet["alt"].min():.0f}m — {df_aemet["alt"].max():.0f}m')
print(f'\nEstaciones sin coordenadas: {df_aemet[["lat","lon"]].isnull().any(axis=1).sum()}')


Valores nulos por columna:
ta            117
tamin         117
tamax         117
prec          252
hr            140
vv           1350
vmax         1403
dv           1325
dmax         1390
pres         7374
pres_nmar    8386
vis          8879
inso         8400
ts           8480
tpr          7311
dtype: int64

Rango temperatura:  -20.0°C — 22.7°C
Media temperatura:  6.7°C

Estaciones con precipitación > 0: 57
Precipitación máxima: 0.6 mm

Rango altitud: 0m — 3092m

Estaciones sin coordenadas: 0


In [5]:
#============================================
# Celda 5 — Explorar estructura WAQI
#============================================
waqi_raw = data['calidad_aire']
ciudades = waqi_raw['ciudades']

print(f'Total ciudades OK:  {waqi_raw["total_ciudades_ok"]}')
print(f'Total errores:      {waqi_raw["total_errores"]}')
print(f'\nKeys de una ciudad:')
print(list(ciudades[0].keys()))
print(f'\nEjemplo ciudad[0]:')
print(json.dumps(ciudades[0], indent=2, ensure_ascii=False))


Total ciudades OK:  30
Total errores:      3

Keys de una ciudad:
['ciudad', 'nombre', 'lat', 'lon', 'timestamp', 'aqi', 'dominante', 'NO2', 'PM25', 'PM10', 'O3', 'SO2', 'CO', 'temperatura', 'humedad', 'presion', 'viento']

Ejemplo ciudad[0]:
{
  "ciudad": "madrid",
  "nombre": "Madrid",
  "lat": 40.4167754,
  "lon": -3.7037902,
  "timestamp": "2026-02-09 14:00:00",
  "aqi": 28,
  "dominante": "o3",
  "NO2": 10.1,
  "PM25": 21,
  "PM10": 13,
  "O3": 28.1,
  "SO2": 0.6,
  "CO": 0.1,
  "temperatura": 10.5,
  "humedad": 99,
  "presion": 1009,
  "viento": 10
}


In [6]:
#============================================
# Celda 6 — Parsear WAQI a DataFrame
#============================================
df_waqi = pd.DataFrame(ciudades)

# Convertir tipos
df_waqi['timestamp'] = pd.to_datetime(df_waqi['timestamp'], errors='coerce')

cols_num = ['lat','lon','aqi','NO2','PM25','PM10','O3','SO2','CO',
            'temperatura','humedad','presion','viento']
for c in cols_num:
    if c in df_waqi.columns:
        df_waqi[c] = pd.to_numeric(df_waqi[c], errors='coerce')

print(f'Shape: {df_waqi.shape}')
print(f'Columnas: {list(df_waqi.columns)}')
df_waqi.head(10)


Shape: (30, 17)
Columnas: ['ciudad', 'nombre', 'lat', 'lon', 'timestamp', 'aqi', 'dominante', 'NO2', 'PM25', 'PM10', 'O3', 'SO2', 'CO', 'temperatura', 'humedad', 'presion', 'viento']


,ciudad,nombre,lat,lon,timestamp,aqi,dominante,NO2,PM25,PM10,O3,SO2,CO,temperatura,humedad,presion,viento
0,madrid,Madrid,40.416775,-3.703790,2026-02-09 14:00:00,28,o3,10.1,21.0,13.0,28.1,0.6,0.1,10.5,99.0,1009.0,10.0
1,barcelona,"Barcelona (Eixample), Catalunya, Spain",41.385343,2.153822,2026-03-30 07:00:00,28,o3,9.2,30.0,14.0,28.1,2.1,0.1,7.2,39.0,1027.1,6.3
2,valencia,"Pista de Silla, València, Valencia, Spain",39.456111,-0.375833,2026-02-13 02:00:00,34,pm25,1.9,34.0,12.0,29.3,1.6,0.1,13.8,63.0,NaN,0.6
3,sevilla,"Ranilla, Sevilla, Spain",37.384250,-5.959620,2026-03-30 08:00:00,36,pm25,21.1,36.0,13.0,25.1,2.6,0.1,9.1,48.2,1028.5,2.6
4,bilbao,"Mazarredo, Bilbao, País Vasco, Spain",43.267500,-2.935200,2026-03-30 06:00:00,38,pm25,23.8,38.0,20.0,NaN,2.6,0.1,8.0,81.0,1034.0,1.5
5,zaragoza,"El Picarral, Zaragoza, Spain",41.670278,-0.871111,2026-03-30 07:00:00,28,o3,4.8,12.0,NaN,27.5,NaN,1.9,7.7,47.0,1028.4,9.3
6,malaga,"Carranque, Malaga, Spain",36.719640,-4.447500,2026-03-30 08:00:00,28,o3,25.2,20.0,13.0,27.6,1.6,0.1,10.0,68.6,1028.0,1.0
7,murcia,"San Basilio Murcia Ciudad, Murcia, Spain",37.993960,-1.144628,2026-03-30 07:00:00,40,pm25,30.2,40.0,25.0,3.6,0.2,0.1,6.6,54.0,1026.7,3.0
8,palma,"foners, Baleares, Spain",39.571250,2.657028,2026-02-09 14:00:00,32,o3,5.1,30.0,15.0,31.7,1.9,0.1,16.1,56.0,1002.7,4.0
9,alicante,"Rabassa, Alacant, Valencia, Spain",38.351111,-0.513889,2026-02-13 04:00:00,29,o3,2.3,9.0,3.0,28.5,1.6,0.1,15.6,10.0,1021.7,2.4


In [7]:
#============================================
# Celda 7 — Análisis rápido WAQI
#============================================
def nivel_aqi(aqi):
    if pd.isna(aqi):       return 'Sin datos'
    elif aqi <= 50:        return '🟢 Bueno'
    elif aqi <= 100:       return '🟡 Moderado'
    elif aqi <= 150:       return '🟠 Insalubre sensibles'
    elif aqi <= 200:       return '🔴 Insalubre'
    else:                  return '🟣 Muy insalubre'

df_waqi['nivel_aqi'] = df_waqi['aqi'].apply(nivel_aqi)

print('Ciudades por nivel AQI:')
print(df_waqi['nivel_aqi'].value_counts().to_string())

print(f'\nTop 5 peor AQI:')
print(df_waqi.nlargest(5, 'aqi')[['nombre','aqi','dominante','PM25','PM10','O3']].to_string(index=False))

print(f'\nTop 5 mejor AQI:')
print(df_waqi.nsmallest(5, 'aqi')[['nombre','aqi','dominante','PM25','PM10','O3']].to_string(index=False))

print(f'\nValores nulos por columna:')
print(df_waqi.isnull().sum()[df_waqi.isnull().sum() > 0])


Ciudades por nivel AQI:
nivel_aqi
🟢 Bueno       29
🟡 Moderado     1

Top 5 peor AQI:
                                  nombre  aqi dominante  PM25  PM10   O3
           Constitución, Asturias, Spain   55      pm25  55.0  19.0 17.1
                         Burgos I, Spain   45      pm10   9.0  45.0 16.3
                       Nativitas, Mexico   41      pm10   NaN  41.0 18.3
San Basilio Murcia Ciudad, Murcia, Spain   40      pm25  40.0  25.0  3.6
                 La Orden, Huelva, Spain   39      pm25  39.0  38.0 22.4

Top 5 mejor AQI:
                                     nombre  aqi dominante  PM25  PM10   O3
     Enseada do Suá, Espírito Santo, Brazil    4        o3   NaN   NaN  3.9
Girona (Escola de Música), Catalunya, Spain   12      pm10  13.0  12.0 16.0
         Lope de Vega, Vigo, Galicia, Spain   13      pm25  13.0   9.0  6.2
     Arco de ladrillo II, Valladolid, Spain   17      pm25  17.0   8.0 23.6
         Santander Centro, Cantabria, Spain   20        o3   NaN  10.0 19.5

Va

In [8]:
#============================================
# Celda 8 — Resumen calidad de datos
#============================================
resumen = pd.DataFrame([
    {
        'tabla':     'aemet_estaciones',
        'filas':     len(df_aemet),
        'columnas':  len(df_aemet.columns),
        'periodos':  df_aemet['fint'].nunique(),
        'nulos_%':   round(df_aemet.isnull().sum().sum() / (df_aemet.shape[0] * df_aemet.shape[1]) * 100, 1)
    },
    {
        'tabla':     'waqi_ciudades',
        'filas':     len(df_waqi),
        'columnas':  len(df_waqi.columns),
        'periodos':  df_waqi['timestamp'].nunique(),
        'nulos_%':   round(df_waqi.isnull().sum().sum()  / (df_waqi.shape[0]  * df_waqi.shape[1])  * 100, 1)
    },
])
print(resumen.to_string(index=False))


           tabla  filas  columnas  periodos  nulos_%
aemet_estaciones  10059        22        13     24.9
   waqi_ciudades     30        18        11      3.3


In [9]:
#============================================
# Celda 9 — Guardar parquets
#============================================
os.makedirs('../../output/ambiental/01-raw', exist_ok=True)

df_aemet.to_parquet('../../output/ambiental/01-raw/raw_aemet_estaciones.parquet', index=False)
print(f'✅ Guardado: output/ambiental/01-raw/raw_aemet_estaciones.parquet ({len(df_aemet)} filas)')

df_waqi.to_parquet('../../output/ambiental/raw_waqi_ciudades.parquet', index=False)
print(f'✅ Guardado: output/ambiental/raw_waqi_ciudades.parquet ({len(df_waqi)} filas)')

print('\n🎉 Todos los parquets ambiental guardados en output/')


✅ Guardado: output/ambiental/01-raw/raw_aemet_estaciones.parquet (10059 filas)


✅ Guardado: output/ambiental/raw_waqi_ciudades.parquet (30 filas)

🎉 Todos los parquets ambiental guardados en output/
